# Report + Business Stakeholder Review — Validation

Final human-review notebook for Sean Step 6. Run this **last**, after the formatters / agents / router notebooks.

Plan: `project_planning/sean_step_artifacts/Report_Business_Review_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/Report_Business_Review_Checklist.md`

This notebook does the closeout work:
- runs the focused Step 6 test suite
- runs an architecture-boundary check on `tools/reporting.py`
- prints a final summary of files added / changed
- surfaces the remaining limitations the user should know about before merging

In [ ]:
from pathlib import Path
import inspect
import subprocess

from multi_agent_ds.tools import reporting as reporting_module
from multi_agent_ds.agents import report_writer as report_agent
from multi_agent_ds.agents import business_stakeholder as biz_agent
from multi_agent_ds.orchestration import router as router_module

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root.")

ROOT = resolve_repo_root()
print("Repo root:", ROOT)

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


## 1. Architecture-boundary check

Confirm the layer rules per `project_planning/ARCHITECTURE.md`:
- `tools/reporting.py` does NOT import from `agents/`, `orchestration/`, or `workflows/`
- `agents/report_writer.py` imports `tools.reporting` (allowed) and `adapters.llm.build_adapter` (allowed)
- `agents/business_stakeholder.py` import surface unchanged in shape (additive: `BusinessReviewOutput`, formatters)

In [ ]:
FORBIDDEN_IN_TOOLS = (
    "multi_agent_ds.agents",
    "multi_agent_ds.orchestration",
    "multi_agent_ds.workflows",
)

def import_lines(module) -> list[str]:
    return [line for line in inspect.getsource(module).splitlines() if line.startswith(("import ", "from "))]

for name, module, forbidden in [
    ("tools/reporting.py", reporting_module, FORBIDDEN_IN_TOOLS),
]:
    print(f"--- {name} imports ---")
    for line in import_lines(module):
        print(" ", line)
    bad = [line for line in import_lines(module) if any(f in line for f in forbidden)]
    assert not bad, f"{name} has forbidden imports: {bad}"
    print(f"  -> clean: no forbidden imports\n")

print("--- agents/report_writer.py imports ---")
for line in import_lines(report_agent):
    print(" ", line)

print("\n--- agents/business_stakeholder.py imports ---")
for line in import_lines(biz_agent):
    print(" ", line)


## 2. Focused Step 6 test run

All 24 new tests must pass.

In [ ]:
run_pytest(["tests/test_report_business_review.py", "-v"])


## 3. Files added / changed in Step 6

Cross-check against the plan's **Implementation Steps** section.

In [ ]:
step6_paths = [
    ("src/multi_agent_ds/orchestration/state.py", "added should_revise_report + report_iteration"),
    ("src/multi_agent_ds/tools/reporting.py", "added 4 pure prompt-context formatters"),
    ("src/multi_agent_ds/agents/report_writer.py", "NEW: report_writer_node(state, mode='generate')"),
    ("src/multi_agent_ds/agents/business_stakeholder.py", "added report_review mode"),
    ("src/multi_agent_ds/orchestration/router.py", "added route_after_business_review"),
    ("config/prompts.yaml", "new report_writer section + revised business_stakeholder.report_review"),
    ("config/workflows.yaml", "new workflows.report.max_iterations entry"),
    ("tests/test_report_business_review.py", "NEW: 24 focused tests for the slice"),
    ("notebooks/Report_Business_Review_Formatters_Review.ipynb", "NEW: human-review notebook"),
    ("notebooks/Report_Business_Review_Agents_Review.ipynb", "NEW: human-review notebook"),
    ("notebooks/Report_Business_Review_Router_Review.ipynb", "NEW: human-review notebook"),
    ("notebooks/Report_Business_Review_Validation.ipynb", "NEW: this notebook"),
]

for path, why in step6_paths:
    full = ROOT / path
    marker = "OK" if full.exists() else "MISSING"
    print(f"  [{marker}] {path:62s} — {why}")


## 4. Remaining limitations to surface in PR / commit body

Be explicit about what this slice does NOT do:

- `state['evaluation_result']` is owned by Jonathan's evaluation pipeline (Jonathan Step 2 / BUILD_PLAN Step 9). The report writer treats it as **optional** and includes a caveat block when missing. End-to-end with real evaluation context will not be exercised until Jonathan's pipeline lands.
- The graph wiring in `orchestration/graph.py` is NOT updated by this slice. Adding `report_writer` and the new business-review node into the StateGraph is a separate slice (lives outside Step 6 scope per the plan).
- No call to the real OpenAI API was made — every test uses a mocked adapter via `build_adapter` patching. A live smoke test against the real API is a manual step (run `report_writer_node` against a populated state with `OPENAI_API_KEY` set).
- The Streamlit `app.py` does NOT yet surface the experiment report or the business review verdict. Display wiring is a future slice.
- The slice was committed onto the Step 5 branch (`feature/step-10-ml-modeler-reviewer`) rather than a fresh Step 6 branch off main, per explicit user direction.

## 5. Sign-off

If everything above passed and you are satisfied with the slice, tick the **Step 5 (Final Validation)** human-review boxes in `Report_Business_Review_Checklist.md` and the closeout is complete.